# Minimal Unsloth GRPO on Colab with a remote OpenEnv Space

This notebook is intentionally similar to the 2048 notebook pattern:
- training runs locally inside Colab
- the environment is accessed remotely through a Hugging Face Space
- the reward function is defined in notebook code by replaying actions against that remote env
- prompt / action / conclusion formatting mirrors the repo logic without importing the repo training script

Default remote env: `Ev3Dev/hackathon`

**Runtime**: Enable a GPU in Colab: Runtime -> Change runtime type -> GPU.

In [ ]:
# 1. Clone the repo for lightweight client / model definitions only
REPO_URL = "https://github.com/mhtruong1031/OpenENV-Hackathon.git"  # or your fork
REPO_DIR = "OpenENV-Hackathon"

!git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

fatal: destination path 'OpenENV-Hackathon' already exists and is not an empty directory.
/content/OpenENV-Hackathon


In [ ]:
# 2. Optimized Unsloth and Training Installation
import os, importlib.util
!pip install --upgrade -qqq uv

# Install core Unsloth stack from source as requested
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil = "pillow"

    !uv pip install -qqq --system \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# Install additional libraries with SciPy pin to avoid the _is_numpy_array error
!uv pip install -qqq --system \
    "scipy==1.13.1" \
    "openenv-core[core]>=0.2.0" \
    "pydantic>=2,<3" \
    "datasets" \
    "accelerate>=1.13,<2" \
    "peft>=0.15,<1" \
    "transformers>=4.57.1,<4.58" \
    "trl>=0.29,<0.30" \
    "matplotlib"

In [ ]:
# 3. Import repo reward helpers, but keep the environment remote
import inspect
import json
import random
import sys
import os
from pathlib import Path
from typing import Any, Dict, List

# Ensure we are in the repo directory even after a runtime restart
REPO_DIR = "OpenENV-Hackathon"
if os.path.exists(REPO_DIR) and os.getcwd().split('/')[-1] != REPO_DIR:
    os.chdir(REPO_DIR)

# Unsloth must be imported before trl / transformers / peft.
import unsloth  # noqa: F401
import torch
from unsloth import FastLanguageModel, PatchFastRL

sys.path.insert(0, str(Path.cwd()))

try:
    from client import BioExperimentEnv
    from models import ActionType, ExperimentAction
    from training_script import (
        INVALID_ACTION_PENALTY,
        ENVIRONMENT_ERROR_PENALTY,
        OpenEnvReward,
        build_training_prompt,
        build_experiment_action,
        decode_history_actions,
        pick_action,
        save_training_plots,
    )
except ModuleNotFoundError as e:
    print(f"Error: {e}. Current Directory: {os.getcwd()}")
    print("Please ensure Cell #1 (git clone) was executed successfully.")
    raise

MAX_COMPLETION_TOKENS = 160
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

def hf_space_repo_to_base_url(repo_id: str) -> str:
    owner, space_name = repo_id.split("/", 1)
    return f"https://{owner.lower().replace('_', '-')}-{space_name.lower().replace('_', '-')}.hf.space"

def build_remote_prompt_examples(
    base_url: str,
    dataset_episodes: int,
    rollout_steps: int,
    seed: int,
) -> List[Dict[str, str]]:
    rng = random.Random(seed)
    examples: List[Dict[str, str]] = []
    for _ in range(dataset_episodes):
        with BioExperimentEnv(base_url=base_url) as env:
            result = env.reset()
            obs = result.observation
            history_actions: List[ExperimentAction] = []
            for step_idx in range(rollout_steps):
                if obs.done: break
                next_action = build_experiment_action(
                    action_type=pick_action("heuristic", step_idx, [action.action_type for action in history_actions]),
                    discovered_markers=obs.discovered_markers,
                    candidate_mechanisms=obs.candidate_mechanisms,
                    conditions=obs.task.conditions,
                )
                examples.append({
                    "prompt": build_training_prompt(obs),
                    "history_actions": json.dumps([action.model_dump() for action in history_actions]),
                    "reference_action": json.dumps(next_action.model_dump()),
                    "problem_statement": obs.task.problem_statement,
                    "episode_tag": f"remote-{rng.randrange(10**9):09d}",
                })
                history_actions.append(next_action)
                result = env.step(next_action)
                obs = result.observation
                if result.done: break
    return examples

def build_grpo_config(**overrides: Any):
    from trl import GRPOConfig
    supported = set(inspect.signature(GRPOConfig.__init__).parameters)
    config_kwargs = {
        "output_dir": overrides["output_dir"],
        "learning_rate": overrides["learning_rate"],
        "per_device_train_batch_size": overrides["per_device_train_batch_size"],
        "gradient_accumulation_steps": overrides["gradient_accumulation_steps"],
        "num_generations": overrides["num_generations"],
        "max_completion_length": overrides["max_completion_length"],
        "num_train_epochs": overrides["num_train_epochs"],
        "logging_steps": overrides["logging_steps"],
        "save_steps": overrides["save_steps"],
        "bf16": overrides["bf16"],
        "fp16": overrides["fp16"],
        "report_to": "none",
        "remove_unused_columns": False,
    }
    if "max_prompt_length" in supported: config_kwargs["max_prompt_length"] = overrides["max_prompt_length"]
    return GRPOConfig(**{k: v for k, v in config_kwargs.items() if k in supported})

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
Path("artifacts").mkdir(exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

CUDA: True Tesla T4


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# 4. Config + collect prompt states from the remote Space
SPACE_REPO_ID = "Ev3Dev/hackathon"
SPACE_BASE_URL = hf_space_repo_to_base_url(SPACE_REPO_ID)
# If your Space has a custom domain, replace SPACE_BASE_URL manually.

MODEL_ID = "unsloth/Qwen3-4B"
OUTPUT_DIR = "artifacts/grpo-unsloth-unsloth/Qwen3-4B-remote-space"

DATASET_EPISODES = 24
ROLLOUT_STEPS = 8
NUM_GENERATIONS = 2

# Increased MAX_SEQ_LENGTH to 2048 to provide a buffer for GRPO internal masks
MAX_PROMPT_LENGTH = 1024
MAX_SEQ_LENGTH = 2048
MAX_COMPLETION_TOKENS = 160

PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LEARNING_RATE = 2e-6
NUM_TRAIN_EPOCHS = 1.5

LOGGING_STEPS = 1
SAVE_STEPS = 50
SEED = 42

LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

examples = build_remote_prompt_examples(
    base_url=SPACE_BASE_URL,
    dataset_episodes=DATASET_EPISODES,
    rollout_steps=ROLLOUT_STEPS,
    seed=SEED,
)

reward_fn = OpenEnvReward(
    reward_backend="remote",
    base_url=SPACE_BASE_URL,
    invalid_action_penalty=INVALID_ACTION_PENALTY,
    environment_error_penalty=ENVIRONMENT_ERROR_PENALTY,
)

print("Remote env:", SPACE_BASE_URL)
print("Prompt states:", len(examples))
print("Sample prompt preview:\n")
print(examples[0]["prompt"][:2000])

Remote env: https://ev3dev-hackathon.hf.space
Prompt states: 192
Sample prompt preview:

You are an expert biologist planning a single-cell experiment pipeline.

At each turn you see the experiment state and must pick the next scientifically justified step.

Environment-specific reasoning rules:
  - Each successful action already returns summarized scientific evidence, so repeated sampling or repeated analysis is not the default.
  - Repeat a step only when the task demands it or when prior outputs show poor quality, insufficient yield, unresolved batch effects, or another clear failure mode.
  - The available tool and assay lists are already filtered to the current task modality, so prefer them over inventing incompatible methods.
  - Hard scientific prerequisites are enforced by the environment, so invalid pipeline orderings will be blocked.

Action guidance:
  - collect_sample: Wet-lab entry point. One successful collection usually provides enough material to continue unless the out

In [ ]:
# 5. Local GRPO training in Colab, remote env for rewards
from datasets import Dataset
from trl import GRPOTrainer

PatchFastRL("GRPO", FastLanguageModel)
train_dataset = Dataset.from_list(examples)

bf16 = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)()) if torch.cuda.is_available() else False
runtime_dtype = torch.bfloat16 if bf16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=runtime_dtype,
    load_in_4bit=True,
)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=SEED,
)

training_args = build_grpo_config(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_TOKENS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    bf16=bf16,
    fp16=torch.cuda.is_available() and not bf16,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

for attr in ("image_token_id", "vision_start_token_id", "vision_end_token_id"):
    if not hasattr(trainer, attr):
        setattr(trainer, attr, None)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
plot_paths = save_training_plots(trainer.state.log_history, OUTPUT_DIR)

result = {
    "trainer": trainer,
    "plot_paths": plot_paths,
    "output_dir": OUTPUT_DIR,
}
print("Saved to:", OUTPUT_DIR)
print("Plots:", plot_paths)

Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=15840) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 192 | Num Epochs = 2 | Total steps = 72
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 40960}. If this is not desired, please set these values explicitly.
Could not estimate the number of tokens of the input, fl

Step,Training Loss,reward,reward_std,train,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,tools / call_frequency,tools / failure_frequency,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,cispo_clip_ratio,rewards / openenv_reward / mean,rewards / openenv_reward / std
1,0.000000,-0.921342,1.637726,0,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,-0.000000,0,-0.921342,1.637726
2,0.000000,-1.259203,1.582013,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,-0.000000,No Log,-1.259203,1.582013
3,0.000000,-1.551267,1.081424,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000013,No Log,-1.551267,1.081424
4,0.000000,-0.888143,1.843477,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000014,No Log,-0.888143,1.843477
5,0.000000,-0.796444,1.663020,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000013,No Log,-0.796444,1.663020
6,0.000000,-0.292611,1.574152,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000015,No Log,-0.292611,1.574152
7,0.000000,-1.175128,1.527395,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000019,No Log,-1.175128,1.527395
8,0.000000,-1.197214,1.487097,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000017,No Log,-1.197214,1.487097
9,0.000000,-1.448854,1.777346,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000014,No Log,-1.448854,1.777346
10,0.000000,-1.221204,1.442134,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000020,No Log,-1.221204,1.442134


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Step,Training Loss,reward,reward_std,train,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,tools / call_frequency,tools / failure_frequency,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,cispo_clip_ratio,rewards / openenv_reward / mean,rewards / openenv_reward / std
1,0.000000,-0.921342,1.637726,0,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,-0.000000,0,-0.921342,1.637726
2,0.000000,-1.259203,1.582013,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,-0.000000,No Log,-1.259203,1.582013
3,0.000000,-1.551267,1.081424,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000013,No Log,-1.551267,1.081424
4,0.000000,-0.888143,1.843477,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000014,No Log,-0.888143,1.843477
5,0.000000,-0.796444,1.663020,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000013,No Log,-0.796444,1.663020
6,0.000000,-0.292611,1.574152,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000015,No Log,-0.292611,1.574152
7,0.000000,-1.175128,1.527395,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000019,No Log,-1.175128,1.527395
8,0.000000,-1.197214,1.487097,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000017,No Log,-1.197214,1.487097
9,0.000000,-1.448854,1.777346,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000014,No Log,-1.448854,1.777346
10,0.000000,-1.221204,1.442134,No Log,160.000000,160.000000,160.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,No Log,No Log,0.000020,No Log,-1.221204,1.442134


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# 6. (Optional) Show curves and sanity-check the repo reward wrapper
from IPython.display import Image, display

sample_reward = reward_fn(
    completions=[[{"role": "assistant", "content": examples[0]["reference_action"]}]],
    history_actions=[examples[0]["history_actions"]],
)[0]
print("Sample reward for reference action:", sample_reward)

for name, path in (result.get("plot_paths") or {}).items():
    print(name, path)
    display(Image(filename=path))